In [1]:
import time
import jax
import jax.numpy as jnp
import grain.python as grain
import optax
from flax import nnx
import numpy as np
import pandas as pd
import bpe_tokenizer as bpe
from models import picoGPT
from training import train_step,validation_step,compute_loss,EarlyStopping
from sklearn.model_selection import train_test_split
import orbax.checkpoint as ocp
from Inference import generate_poem
import os
from baseline import NGramModel,evaluate_ngram_perplexity

In [2]:
# g++ -O3 -Wall -shared -std=c++14 -fPIC $(python3 -m pybind11 --includes) "/mnt/c/Users/Aneesh Shastri/Documents/GitHub/EpochCoreTasks/Task1/Subtask3/Custom_Tokenizers/bpe_tokenizer.cpp" -o"/mnt/c/Users/Aneesh Shastri/Documents/GitHub/EpochCoreTasks/Task1/Subtask3/bpe_tokenizer"$(python3-config --extension-suffix)


In [3]:
poems=pd.read_csv('poetry.csv')
poems.head()
poems[poems.isna().any(axis=1)]


,author,content,poem name,age,type
354,RICHARD ALDINGTON,The ancient songs\r\nPass deathward mournfully...,NaN,Modern,Mythology & Folklore
489,RICHARD ALDINGTON,The ancient songs\r\nPass deathward mournfully...,NaN,Modern,Nature


In [4]:

poems.fillna('Unknown',inplace=True)
corpus=" Author: ".join(poems['author'].astype(str))+" Name: ".join(poems['poem name'].astype(str))+"\n".join(poems['content'].astype(str))+"Era: ".join(poems['age'].astype(str))+"Style: ".join(poems['type'].astype(str))
Tokenizer=bpe.BPETokenizer()
Tokenizer.train(corpus,num_merges=20_000)
Tokenizer.save("Saved_Models/Tokenizer.bin")

In [5]:
print(Tokenizer.get_vocab_size())
Tokenizer.encode("era")

20256


[261, 97]

In [6]:

def build_dataset(csv_path, tokenizer, max_len=256):
    df = pd.read_csv(csv_path)
    
    vocab_size = tokenizer.get_vocab_size()
    eos_id = vocab_size
    pad_id = vocab_size + 1
    
    all_sequences = []
    all_masks = []
    
    for _, row in df.iterrows():
        meta_text = (
            f"Author: {row['author']}\n"
            f"Era: {row['age']}\n"
            f"Type: {row['type']}\n"
            f"Title: {row['poem name']}\n\n"
        )
        meta_tokens = tokenizer.encode(meta_text)
        content_tokens = tokenizer.encode(row['content'])
        
        M = len(meta_tokens)
        C = (max_len + 1) - M # Total space remaining for content and EOS
        
        if C <= 1:
            continue # Metadata alone consumes the context window
            
        i = 0
        eos_appended = False
        
        while not eos_appended:
            remaining = len(content_tokens) - i
            
            if remaining == 0 and i == 0:
                # Edge case: Empty poem content
                seq = meta_tokens + [eos_id]
                mask = [0] * M + [1]
                eos_appended = True
                chunk_len = 0
            elif remaining + 1 <= C:
                # Final chunk: The remaining content and EOS fit perfectly
                chunk = content_tokens[i : i + remaining]
                seq = meta_tokens + chunk + [eos_id]
                mask = [0] * M + [1] * len(chunk) + [1]
                eos_appended = True
                chunk_len = len(chunk)
            else:
                # Intermediate chunk: Space is exceeded.
                # If remaining == C exactly, we take C - 1 to leave at least 1 token
                # for the final chunk so it isn't just [Meta] + [EOS].
                take = C if remaining > C else C - 1
                chunk = content_tokens[i : i + take]
                seq = meta_tokens + chunk
                mask = [0] * M + [1] * len(chunk)
                chunk_len = len(chunk)
                
            # Pad the sequence if necessary
            pad_length = (max_len + 1) - len(seq)
            if pad_length > 0:
                seq.extend([pad_id] * pad_length)
                mask.extend([0] * pad_length)
                
            all_sequences.append(seq)
            all_masks.append(mask)
            
            i += chunk_len
            
    dataset_np = np.array(all_sequences, dtype=np.int32)
    mask_np = np.array(all_masks, dtype=np.int32)
    
    return jax.device_put(dataset_np), jax.device_put(mask_np), eos_id, pad_id


def sample_batch(dataset_2d, mask_2d, rng_key, batch_size):
    num_samples = dataset_2d.shape[0]
    indices = jax.random.randint(rng_key, shape=(batch_size,), minval=0, maxval=num_samples)
    
    batch_seq = dataset_2d[indices, :]
    batch_mask = mask_2d[indices, :]
    
    return batch_seq, batch_mask

In [7]:
X, mask,eos_id,pad_id=build_dataset(csv_path="poetry.csv",tokenizer=Tokenizer)

X_train_cpu, X_temp, mask_train_cpu, mask_temp = train_test_split(X, mask, test_size=0.2, random_state=42)
X_val_cpu, X_test_cpu, mask_val_cpu, mask_test_cpu = train_test_split(X_temp, mask_temp, test_size=0.5, random_state=42)

X_train = jax.device_put(X_train_cpu)
mask_train = jax.device_put(mask_train_cpu)
X_val = jax.device_put(X_val_cpu)
mask_val = jax.device_put(mask_val_cpu)
X_test = jax.device_put(X_test_cpu)
mask_test = jax.device_put(mask_test_cpu)

In [8]:

TOTAL_VOCAB_SIZE=Tokenizer.get_vocab_size()+2
for n_val in [1,2, 3]:
    print(f"\n--- Training Markov Model (N={n_val}) ---")
    t0 = time.time()
    
    baseline_model = NGramModel(
        n=n_val, 
        vocab_size=TOTAL_VOCAB_SIZE, 
        pad_id=pad_id, 
        eos_id=eos_id
    )
    
    # Train the model (Building the dictionary)
    baseline_model.fit(X_train_cpu)
    print(f"Training Complete. Time: {time.time() - t0:.2f}s")
    
    #Evaluate Perplexity
    val_perplexity = evaluate_ngram_perplexity(baseline_model,X_val_cpu)
    print("VAL PERPLEXITY=" ,val_perplexity)
    test_perplexity = evaluate_ngram_perplexity(baseline_model,X_test_cpu)
    print("TEST PERPLEXITY=" ,test_perplexity)
    # Generate a sequence
    
    metadata = {
        "author": "WILLIAM SHAKESPEARE",
        "era": "Renaissance",
        "type": "Mythology & Folklore",
        "title": "The Phoenix and the Turtle"
    }
    
    prompt_text = (
        f"Author: {metadata['author']}\n"
        f"Era: {metadata['era']}\n"
        f"Type: {metadata['type']}\n"
        f"Title: {metadata['title']}\n\n"
    )
    
    prompt_tokens = Tokenizer.encode(prompt_text)
    
    output_tokens = baseline_model.generate(prompt_tokens)
    print("\nGenerated Output:")
    print(Tokenizer.decode(output_tokens))


--- Training Markov Model (N=1) ---
Training Complete. Time: 0.54s
VAL PERPLEXITY= 929.082958661863
TEST PERPLEXITY= 997.1184725316165

Generated Output:
Author: WILLIAM SHAKESPEARE
Era: Renaissance
Type: Mythology & Folklore
Title: The Phoenix and the Turtle

   in megade badge of loveth should we that ensuestwoetle towne   The: Love
To se smooth tongue was indulgently  inness falls that snap-out common song
As water loves prise loss and wef   It Hath My Picture Losteve narrows of better; 
About the grove,
Tive Rems ac bonus
 all and shinver H. in the more.

Lost, being elemented t if unfit
First faint:
Her tress a-cookin']
 . LAWRENCE
Era: Modern
Type Estate of darkness; 
Ariseyes of the: many wars docks on my Goddesse she: Love, more the huthor: JOHN DONNE
'Elpenor, our wit charge nor knows
The, from salt flood
Than those pure,   in made, out your wisdoms golden ho long, his men my heart, and watched the    t and  tls bosom is no from fair, and courtly she creeper leaves
 love. The

In [9]:
d_k=32
num_heads=4
num_layers=4
max_seq_len=256
rngs  = nnx.Rngs(params=0, dropout=1)
lr=5e-4
batch_size=64


schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0, peak_value=lr,
        warmup_steps=200, decay_steps=5000, end_value=1e-6,
    )


model=picoGPT(vocab_size=TOTAL_VOCAB_SIZE,d_model=d_k*num_heads,num_heads=num_heads,num_layers=num_layers,max_seq_len=max_seq_len,rngs=rngs,dropout_rate=0.3)
optimizer = nnx.Optimizer(model, optax.adamw(schedule, weight_decay=1e-4), wrt=nnx.Param)



In [10]:
import jax
import jax.numpy as jnp

def get_permutation(num_samples: int, batch_size: int, epoch: int) -> jax.Array:
    # Seed PRNG with epoch for perfect reproducibility
    rng_key = jax.random.PRNGKey(epoch)
    
    # Generate shuffled indices
    indices = jax.random.permutation(rng_key, jnp.arange(num_samples, dtype=jnp.int32))
    
    # Calculate valid sequence limit to drop the remainder
    valid_samples = (num_samples // batch_size) * batch_size
    
    return indices[:valid_samples]

In [11]:
@jax.jit(static_argnames="batch_size")
def get_batch(X_dataset, mask_dataset, epoch_indices, step: int, batch_size: int):
    # Calculate the starting pointer for this specific step
    start_idx = step * batch_size
    
    # Extract the pre-shuffled indices for this batch
    # jax.lax.dynamic_slice ensures static tensor shape requirements for XLA
    batch_idx = jax.lax.dynamic_slice(epoch_indices, (start_idx,), (batch_size,))
    
    # Gather and return the data mapped to the dictionary format expected by train_step
    return  (X_dataset[batch_idx], mask_dataset[batch_idx])
    

In [12]:
patience=1
epochs=100 

es = EarlyStopping(patience=patience)
steps_per_epoch=X_train.shape[0]//batch_size
for epoch in range(epochs):
    t0 = time.time()
    train_losses = []
    epoch_indices=get_permutation(X_train.shape[0],batch_size,epoch)
    model.train()
    for step in range(steps_per_epoch):
        batch = get_batch(X_train, mask_train, epoch_indices,step,batch_size)
        loss = train_step(model, optimizer, batch, pad_id)
        train_losses.append(loss)

    model.eval()
    val_batch = (X_val, mask_val)
    val_loss = validation_step(model, val_batch,pad_id)
    print(f"Epoch {epoch+1} | Train Loss: {np.mean(train_losses):.4f} | Val Loss: {val_loss:.4f} | Time: {time.time()-t0:.2f}s")
    if es.step(val_loss, model):
        print("Early stopping triggered.")
        break

if es.best_state is not None:
    nnx.update(model, es.best_state)
    print("Restored best model state.")

Epoch 1 | Train Loss: 10.3602 | Val Loss: 10.1741 | Time: 64.39s
Epoch 2 | Train Loss: 10.1108 | Val Loss: 9.5608 | Time: 0.89s
Epoch 3 | Train Loss: 9.6476 | Val Loss: 8.9616 | Time: 0.90s
Epoch 4 | Train Loss: 9.0800 | Val Loss: 8.4657 | Time: 0.90s
Epoch 5 | Train Loss: 8.5668 | Val Loss: 8.0156 | Time: 0.90s
Epoch 6 | Train Loss: 8.0899 | Val Loss: 7.5669 | Time: 0.91s
Epoch 7 | Train Loss: 7.6033 | Val Loss: 7.1209 | Time: 0.90s
Epoch 8 | Train Loss: 7.1393 | Val Loss: 6.6835 | Time: 0.90s
Epoch 9 | Train Loss: 6.6989 | Val Loss: 6.3220 | Time: 0.90s
Epoch 10 | Train Loss: 6.3639 | Val Loss: 6.0696 | Time: 0.90s
Epoch 11 | Train Loss: 6.1284 | Val Loss: 5.9158 | Time: 0.91s
Epoch 12 | Train Loss: 5.9795 | Val Loss: 5.8334 | Time: 0.90s
Epoch 13 | Train Loss: 5.8877 | Val Loss: 5.7814 | Time: 0.92s
Epoch 14 | Train Loss: 5.8290 | Val Loss: 5.7274 | Time: 0.91s
Epoch 15 | Train Loss: 5.7672 | Val Loss: 5.6776 | Time: 0.91s
Epoch 16 | Train Loss: 5.7032 | Val Loss: 5.6049 | Time: 0.9

In [13]:
state = nnx.state(model)
ckpt_dir_t = os.path.abspath("Saved_Models/picoGPT")

checkpointer=ocp.StandardCheckpointer()
checkpointer.save(ckpt_dir_t, state, force=True)

In [14]:

def evaluate_test_perplexity(model, X_test, mask_test, pad_id):
    model.eval()
    test_batch = (X_test, mask_test)
    test_loss = validation_step(model, test_batch, pad_id)
    test_perplexity = np.exp(float(test_loss))
    
    return test_perplexity

evaluate_test_perplexity(model,X_test,mask_test,pad_id)

np.float64(130.3833695069029)

In [15]:

metadata = {
    "author": "WILLIAM SHAKESPEARE",
    "era": "Renaissance",
    "type": "Mythology & Folklore",
    "title": "The Phoenix and the Turtle"
}

generated_text = generate_poem(model, Tokenizer, metadata,content="", temperature=0.5,max_new_tokens=1000)
print(generated_text)

Then,
That sot, a day.

Then heaves they gay notse,
And I have low in the bember
The wind with the that his love;
But thou had sigh of her eyes, and all my heart,
Handew, sing, and a thought.
O sing that yougelexteth,
I am which I gadefore.

She then their beauty shememeemse to the world is me;
Which to much your we same reely.
Or heavemeee ree to come as for
The world is sembse:
And a smile not is to dear,
If themed to be in the rems to me,
Where whitefeteme, you,
Grpsemselfmersee
As to so melsee ground,
To guch
And but that to semest.

Will wisy's go more did le,
That in thy heart of thememer.

Who love his fair will be, till sot
A foot, and how,
But which my head to with thew,
Nor thou all her mind that we, and I have note,
Her mew, if this shan, and like my love,
He, and though my face in themse,
Doo of the other they bemdle;
        And theeen out.
